<a href="https://colab.research.google.com/github/DemonWillCode/Demo-Dashboard/blob/main/Delhi_final_cleaning%2BModel_googlecollab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

# ML
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Encoding
from sklearn.preprocessing import OneHotEncoder

In [ ]:
df = pd.read_csv("delhi_cleaned.csv")

print(df.head())
print(df.info())

      Price  Area          Location  No. of Bedrooms  Resale  \
0  10500000  1200  Sector 10 Dwarka                2       1   
1   6000000  1000       Uttam Nagar                3       0   
2  15000000  1350      Sarita Vihar                2       1   
3   2500000   435       Uttam Nagar                2       0   
4   5800000   900        Dwarka Mor                3       0   

   LandscapedGardens  IndoorGames  Intercom  SportsFacility  ClubHouse  \
0                  0            0         1               1          0   
1                  0            0         1               0          0   
2                  0            0         0               0          0   
3                  0            0         1               0          0   
4                  0            0         0               0          0   

   24X7Security  PowerBackup  CarParking  Gasconnection  AC  Wifi  \
0             1            1           1              1   0     0   
1             0            1    

In [ ]:
X = df.drop("Price", axis=1)   # Features
y = df["Price"]                # Target

In [ ]:
X = pd.get_dummies(X, columns=["Location"], drop_first=True)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [ ]:
importance = pd.Series(model.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False)

print(importance.head(10))

Area                                      0.457662
Location_Sunder Nagar                     0.058731
No. of Bedrooms                           0.051518
Location_Saket                            0.037109
Resale                                    0.029712
Location_Greater Kailash                  0.027913
Location_Vasant Kunj                      0.027234
Location_Mansarovar garden                0.027119
Location_Shivalik                         0.025587
Location_Om Enclave Mithapur Extension    0.022003
dtype: float64


In [ ]:
y_pred = model.predict(X_test)

from sklearn.metrics import mean_absolute_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("R2 Score:", r2)

MAE: 10497731.663447902
R2 Score: -0.08525370814087063


In [ ]:
df = df.dropna()


In [ ]:
top_locations = df['Location'].value_counts().nlargest(10).index

df['Location'] = df['Location'].apply(
    lambda x: x if x in top_locations else 'Other'
)

In [ ]:
df = pd.get_dummies(df, columns=['Location'], drop_first=True)

In [ ]:
df = df[df['Price'] < df['Price'].quantile(0.99)]
df = df[df['Area'] < df['Area'].quantile(0.99)]

In [ ]:
import numpy as np
df['Price'] = np.log(df['Price'])

In [ ]:
model = RandomForestRegressor(n_estimators=300, random_state=42)
model.fit(X_train, y_train)

RandomForestRegressor(n_estimators=300, random_state=42)

In [ ]:
y_pred = model.predict(X_test)

from sklearn.metrics import mean_absolute_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("R2 Score:", r2)

MAE: 10430250.047316402
R2 Score: -0.0713990942250784


In [ ]:
print(df.corr(numeric_only=True)['Price'].sort_values(ascending=False))

Price                        1.000000
Area                         0.561796
Location_Other               0.347376
Gasconnection                0.341363
No. of Bedrooms              0.333489
Children'splayarea           0.327386
PowerBackup                  0.300494
SportsFacility               0.279736
ClubHouse                    0.242202
Location_Vasant Kunj         0.207720
Resale                       0.197740
Total_Amenities              0.184881
Intercom                     0.161973
LandscapedGardens            0.155177
AC                           0.152939
24X7Security                 0.149629
Location_Sector 6 Dwarka     0.127921
Location_Sector 11 Dwarka    0.121510
Location_Sector 10 Dwarka    0.101271
Location_Sector 12 Dwarka    0.090360
CarParking                   0.075057
LiftAvailable                0.065996
IndoorGames                  0.063571
Location_Greater Kailash    -0.034155
Location_Dwarka Mor         -0.150153
Location_Uttam Nagar        -0.191774
Location_Noi

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("delhi_cleaned.csv")

In [ ]:
df.columns = df.columns.str.strip().str.replace(" ", "_")

In [ ]:
# Drop rows where target is missing
df = df.dropna(subset=["Price"])

# Fill numeric columns with median
num_cols = df.select_dtypes(include=np.number).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Fill categorical
df['Location'] = df['Location'].fillna("Unknown")

In [ ]:
# Convert binary columns to 0/1
binary_cols = [
    'Resale','LandscapedGardens','IndoorGames','Intercom',
    'SportsFacility','ClubHouse','24X7Security','PowerBackup',
    'CarParking','Gasconnection','AC','Wifi',
    "Children'splayarea",'LiftAvailable'
]

for col in binary_cols:
    df[col] = df[col].astype(int)

In [ ]:
# Remove extreme values
df = df[df['Price'] < df['Price'].quantile(0.99)]
df = df[df['Area'] < df['Area'].quantile(0.99)]
df = df[df['Area'] > 100]  # remove unrealistic small areas

In [ ]:
# Price per sqft
df['Price_per_sqft'] = df['Price'] / df['Area']

# Amenity score (normalize)
df['Amenity_Score'] = df['Total_Amenities'] / len(binary_cols)

# Bedrooms density
df['Bedroom_Density'] = df['No._of_Bedrooms'] / df['Area']

In [ ]:
top_locations = df['Location'].value_counts().nlargest(15).index

df['Location'] = df['Location'].apply(
    lambda x: x if x in top_locations else 'Other'
)

In [ ]:
df = pd.get_dummies(df, columns=['Location'], drop_first=True)

In [ ]:
df = df.reset_index(drop=True)

print(df.shape)
print(df.head())

(4898, 36)
       Price  Area  No._of_Bedrooms  Resale  LandscapedGardens  IndoorGames  \
0  16.166886  1200                2       1                  0            0   
1  15.607270  1000                3       0                  0            0   
2  16.523561  1350                2       1                  0            0   
3  14.731802   435                2       0                  0            0   
4  15.573369   900                3       0                  0            0   

   Intercom  SportsFacility  ClubHouse  24X7Security  ...  Location_Other  \
0         1               1          0             1  ...           False   
1         1               0          0             0  ...           False   
2         0               0          0             0  ...            True   
3         1               0          0             0  ...           False   
4         0               0          0             0  ...           False   

   Location_Sector 10 Dwarka  Location_Sector 11 Dw

In [ ]:
df.to_csv("cleaned_model_ready.csv", index=False)

In [ ]:
df = df.astype(int)

In [ ]:
df['Price'] = np.log1p(df['Price'])

In [ ]:
y_pred = model.predict(X_test)

# Convert back to original scale
y_test_actual = np.expm1(y_test)
y_pred_actual = np.expm1(y_pred)

from sklearn.metrics import mean_absolute_error, r2_score

print("MAE:", mean_absolute_error(y_test_actual, y_pred_actual))
print("R2:", r2_score(y_test_actual, y_pred_actual))

/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: overflow encountered in expm1
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipykernel_7162/3207588860.py:5: RuntimeWarning: overflow encountered in expm1
  y_pred_actual = np.expm1(y_pred)


ValueError: Input contains infinity or a value too large for dtype('float64').

In [ ]:
y_pred = model.predict(X_test)

# Clip extreme values
y_pred = np.clip(y_pred, 0, 20)

y_test_actual = np.expm1(y_test)
y_pred_actual = np.expm1(y_pred)

/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: overflow encountered in expm1
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [ ]:
print("Min pred:", y_pred.min())
print("Max pred:", y_pred.max())

Min pred: 20.0
Max pred: 20.0


In [ ]:
print(y_train.min(), y_train.max())

2000000 854599999


In [ ]:
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

In [ ]:
X = df.drop("Price", axis=1)
y = df["Price"]

In [ ]:
print("Target range:", y.min(), y.max())
print("Feature variation:\n", X.nunique().head(10))

Target range: 2.70805020110221 2.995732273553991
Feature variation:
 Area                 408
No._of_Bedrooms        8
Resale                 2
LandscapedGardens      2
IndoorGames            2
Intercom               2
SportsFacility         2
ClubHouse              2
24X7Security           2
PowerBackup            2
dtype: int64


In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=42)

In [ ]:
y_pred = model.predict(X_test)

print("Min:", y_pred.min())
print("Max:", y_pred.max())

Min: 2147287.7777777775
Max: 323827626.9427779


In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("R2:", r2)

MAE: 10430250.047316402
R2: -0.0713990942250784


In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score

# Convert back
y_test_actual = np.expm1(y_test)
y_pred_actual = np.expm1(y_pred)

print("MAE:", mean_absolute_error(y_test_actual, y_pred_actual))
print("R2:", r2_score(y_test_actual, y_pred_actual))

/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: overflow encountered in expm1
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipykernel_7162/401257596.py:6: RuntimeWarning: overflow encountered in expm1
  y_pred_actual = np.expm1(y_pred)


ValueError: Input contains infinity or a value too large for dtype('float64').

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("delhi_cleaned.csv")

In [ ]:
# Fix column names
df.columns = df.columns.str.strip().str.replace(" ", "_")

# Drop missing target
df = df.dropna(subset=["Price"])

# Fill numeric nulls
num_cols = df.select_dtypes(include=np.number).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Fill location null
df['Location'] = df['Location'].fillna("Unknown")

In [ ]:
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

In [ ]:
df = df[df['Price'] < df['Price'].quantile(0.99)]
df = df[df['Area'] < df['Area'].quantile(0.99)]
df = df[df['Area'] > 100]

In [ ]:
df['Price_per_sqft'] = df['Price'] / df['Area']
df['Amenity_Score'] = df['Total_Amenities']
df['Bedroom_Density'] = df['No._of_Bedrooms'] / df['Area']

In [ ]:
top_locations = df['Location'].value_counts().nlargest(15).index

df['Location'] = df['Location'].apply(
    lambda x: x if x in top_locations else 'Other'
)

In [ ]:
df = pd.get_dummies(df, columns=['Location'], drop_first=True)

KeyError: "None of [Index(['Location'], dtype='object')] are in the [columns]"

In [ ]:
print(df.columns)

Index(['Price', 'Area', 'No._of_Bedrooms', 'Resale', 'LandscapedGardens',
       'IndoorGames', 'Intercom', 'SportsFacility', 'ClubHouse',
       '24X7Security', 'PowerBackup', 'CarParking', 'Gasconnection', 'AC',
       'Wifi', 'Children'splayarea', 'LiftAvailable', 'Total_Amenities',
       'Price_per_sqft', 'Amenity_Score', 'Bedroom_Density', 'Location_Burari',
       'Location_Dwarka Mor', 'Location_Jamia Nagar', 'Location_Noida',
       'Location_Om Nagar', 'Location_Other', 'Location_Sector 10 Dwarka',
       'Location_Sector 11 Dwarka', 'Location_Sector 12 Dwarka',
       'Location_Sector 19 Dwarka', 'Location_Sector 22 Dwarka',
       'Location_Sector 4 Dwarka', 'Location_Sector 6 Dwarka',
       'Location_Uttam Nagar', 'Location_Vasant Kunj'],
      dtype='object')


In [ ]:
X = df.drop("Price", axis=1)
y = df["Price"]

In [ ]:
df.to_csv("delhi_cleaned_final.csv",index=False)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=400,
    max_depth=25,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

RandomForestRegressor(max_depth=25, n_estimators=400, n_jobs=-1,
                      random_state=42)

In [ ]:
y_pred = model.predict(X_test)

print("Min:", y_pred.min())
print("Max:", y_pred.max())

Min: 2000527.5
Max: 205993500.0


In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("R2:", r2)

MAE: 627814.108877551
R2: 0.9235667943428556


In [ ]:
import joblib

joblib.dump(model, "delhi_price_model.pkl")
joblib.dump(X.columns, "delhi_model_columns.pkl")

print("Saved successfully!")

Saved successfully!


In [ ]:
import os

print(os.listdir())

['.config', 'delhi_price_model.pkl', 'delhi_cleaned.csv', 'delhi_model_columns.pkl', 'cleaned_model_ready.csv', 'sample_data']


In [ ]:
from google.colab import files

files.download("delhi_price_model.pkl")
files.download("delhi_model_columns.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("R2:", r2)

MAE: 627814.108877551
R2: 0.9235667943428556


In [ ]:
from google.colab import files
files.download("delhi_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>